# 05 Regime Overlay Diagnostic Layer

Purpose: test optional, mild regime overlays on constructed alphas from the 04A/04B pipeline.

Scope note: This notebook tests optional regime overlays on constructed alphas. Its outputs are diagnostic and should not automatically feed survivor freeze or portfolio construction.

Scope boundaries:
- Keep 04A alpha formulas unchanged.
- Keep 04B validation outputs unchanged.
- Write only `regime_context_alpha_*` diagnostic construction tables.
- Require Notebook 06 validation before any overlay can be considered for explicit downstream promotion.


## 1. Imports and Config

In [1]:
from pathlib import Path
import gc
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_db_path
from src.regime_context_alpha import (
    build_constructed_alpha_panels,
    build_regime_context_activation_diagnostics,
    build_regime_context_alpha_candidates,
    build_regime_context_diagnostics,
    build_regime_context_quality,
    load_regime_context_inputs,
    select_approved_constructed_alphas,
    select_strongest_constructed_alpha,
)
from src.regime_context_alpha_storage import (
    REGIME_CONTEXT_ALPHA_TABLES,
    save_regime_context_alpha_outputs,
)
from src.run_config import make_run_id, make_run_timestamp

REGIME_CONTEXT_VERSION = "phase5_regime_overlay_alpha_v3"
DB_PATH = get_db_path()

pd.set_option("display.max_columns", 200)
DB_PATH

PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase5_regime_context_alpha")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase5_regime_context_alpha_20260504_185103', '2026-05-04 18:51:03')

## 3. Load regime-context inputs

In [3]:
inputs = load_regime_context_inputs(db_path=DB_PATH)

input_shapes = pd.DataFrame(
    [
        {"input_name": name, "n_rows": len(df), "n_columns": len(df.columns)}
        for name, df in inputs.items()
    ]
)
display(input_shapes)

,input_name,n_rows,n_columns
0,alpha_long,2108880,6
1,alpha_quality,10,13
2,constructed_alpha_wfv_gate,36,18
3,constructed_alpha_wfv_winner_summary,9,10
4,alpha_metadata,10,9
5,alpha_diagnostics,10,16
6,regime_features,2088,11


## 4. Load approved constructed alphas

In [4]:
approved_constructed_alphas = select_approved_constructed_alphas(
    alpha_quality=inputs["alpha_quality"],
    constructed_alpha_wfv_gate=inputs["constructed_alpha_wfv_gate"],
    constructed_alpha_wfv_winner_summary=inputs["constructed_alpha_wfv_winner_summary"],
)
if approved_constructed_alphas.empty:
    raise ValueError("No constructed alpha candidates are approved for validation.")

approved_display_columns = [
    column
    for column in [
        "alpha_name",
        "finite_pct",
        "missing_pct",
        "status",
        "source_alpha_wfv_status",
        "source_alpha_wfv_horizon",
        "source_effective_mean_test_ic",
        "source_persistence_ratio",
    ]
    if column in approved_constructed_alphas.columns
]
display(approved_constructed_alphas[approved_display_columns])

,alpha_name,finite_pct,missing_pct,status,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
0,alpha_hybrid_adaptive_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
1,alpha_rolling_ic_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
2,alpha_regime_blend_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75
3,alpha_decay_aware_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,alpha_smooth_regime_weighted_v2,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
5,alpha_health_weighted_research_v1,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50
6,alpha_equal_weight_research_v1,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
7,alpha_persistence_blend_v2,0.981066,0.018934,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.139545,0.50
8,alpha_diversified_research_v2,0.988729,0.011271,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.125316,0.50


## 5. Load regime features

In [5]:
regime_features = inputs["regime_features"].copy()
regime_features["Date"] = pd.to_datetime(regime_features["Date"], errors="coerce")

regime_columns = [
    "benchmark_vol_regime",
    "benchmark_trend_regime",
    "drawdown_regime",
    "correlation_regime",
]
regime_feature_coverage = pd.DataFrame(
    [
        {
            "regime_column": column,
            "n_rows": len(regime_features),
            "non_null": int(regime_features[column].notna().sum()),
            "non_null_pct": float(regime_features[column].notna().mean()),
            "n_unique": int(regime_features[column].nunique(dropna=True)),
            "first_date": regime_features.loc[regime_features[column].notna(), "Date"].min(),
            "last_date": regime_features.loc[regime_features[column].notna(), "Date"].max(),
        }
        for column in regime_columns
        if column in regime_features.columns
    ]
)
display(regime_feature_coverage)

,regime_column,n_rows,non_null,non_null_pct,n_unique,first_date,last_date
0,benchmark_vol_regime,2088,2068,0.990421,3,2018-01-31,2026-04-23
1,benchmark_trend_regime,2088,1889,0.904693,3,2018-10-16,2026-04-23
2,drawdown_regime,2088,2088,1.000000,2,2018-01-02,2026-04-23
3,correlation_regime,2088,2068,0.990421,3,2018-01-31,2026-04-23


## 6. Select base alpha candidates

In [6]:
alpha_panels = build_constructed_alpha_panels(
    inputs["alpha_long"],
    approved_constructed_alphas,
)

strongest_constructed_alpha = approved_constructed_alphas["alpha_name"].iloc[0]

base_alpha_summary = pd.DataFrame(
    [
        {
            "alpha_name": alpha_name,
            "n_dates": panel.shape[0],
            "n_tickers": panel.shape[1],
            "finite_pct": float(panel.notna().to_numpy().mean()) if panel.size else 0.0,
            "selected_as_strongest": alpha_name == strongest_constructed_alpha,
        }
        for alpha_name, panel in alpha_panels.items()
    ]
).sort_values(["selected_as_strongest", "alpha_name"], ascending=[False, True])

display(base_alpha_summary)

,alpha_name,n_dates,n_tickers,finite_pct,selected_as_strongest
0,alpha_hybrid_adaptive_v3,2088,101,0.980587,True
3,alpha_decay_aware_dynamic_v3,2088,101,0.980587,False
8,alpha_diversified_research_v2,2088,101,0.988729,False
6,alpha_equal_weight_research_v1,2088,101,0.980587,False
5,alpha_health_weighted_research_v1,2088,101,0.980587,False
7,alpha_persistence_blend_v2,2088,101,0.981066,False
2,alpha_regime_blend_dynamic_v3,2088,101,0.980587,False
1,alpha_rolling_ic_dynamic_v3,2088,101,0.980587,False
4,alpha_smooth_regime_weighted_v2,2088,101,0.980587,False


## 7. Build regime overlay candidates

In [7]:
regime_context_alpha_candidates, regime_context_alpha_metadata = build_regime_context_alpha_candidates(
    alpha_panels=alpha_panels,
    approved_alphas=approved_constructed_alphas,
    constructed_alpha_wfv_gate=inputs["constructed_alpha_wfv_gate"],
    regime_features=regime_features,
)

regime_context_alpha_metadata["run_id"] = run_id
regime_context_alpha_metadata["regime_context_version"] = REGIME_CONTEXT_VERSION

created_alpha_summary = pd.DataFrame(
    [
        {
            "alpha_name": alpha_name,
            "n_dates": panel.shape[0],
            "n_tickers": panel.shape[1],
            "finite_pct": float(panel.notna().to_numpy().mean()) if panel.size else 0.0,
        }
        for alpha_name, panel in regime_context_alpha_candidates.items()
    ]
)
display(created_alpha_summary)

,alpha_name,n_dates,n_tickers,finite_pct
0,alpha_hybrid_adaptive_v3__base_passthrough,2088,101,0.980587
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,2088,101,0.980587
2,alpha_hybrid_adaptive_v3__defensive_downscale,2088,101,0.980587
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,2088,101,0.980587
4,alpha_rolling_ic_dynamic_v3__base_passthrough,2088,101,0.980587
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,2088,101,0.980587
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,2088,101,0.980587
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,2088,101,0.980587
8,alpha_regime_blend_dynamic_v3__base_passthrough,2088,101,0.980587
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,2088,101,0.980587


## 8. Build metadata, quality, and diagnostics

In [8]:
regime_context_alpha_quality = build_regime_context_quality(regime_context_alpha_candidates)
regime_context_alpha_quality["run_id"] = run_id
regime_context_alpha_quality["regime_context_version"] = REGIME_CONTEXT_VERSION

regime_context_alpha_diagnostics = build_regime_context_diagnostics(regime_context_alpha_candidates)
regime_context_alpha_diagnostics["run_id"] = run_id
regime_context_alpha_diagnostics["regime_context_version"] = REGIME_CONTEXT_VERSION

regime_context_alpha_activation = build_regime_context_activation_diagnostics(
    metadata=regime_context_alpha_metadata,
    regime_features=regime_features,
    alpha_candidates=regime_context_alpha_candidates,
)
regime_context_alpha_activation["run_id"] = run_id
regime_context_alpha_activation["regime_context_version"] = REGIME_CONTEXT_VERSION

quality_status_counts = (
    regime_context_alpha_quality["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_alphas")
)

display(regime_context_alpha_metadata)
display(regime_context_alpha_quality)
display(regime_context_alpha_diagnostics)
display(regime_context_alpha_activation)

,alpha_name,overlay_type,base_alpha_name,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio,scaling_rule,regime_column,notes,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,base_passthrough,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75,scale=1.00 for all dates; no overlay.,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,mild_regime_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,defensive_downscale,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...","benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,volatility_stress_scaled,alpha_hybrid_adaptive_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75,"scale=0.75 during HIGH_VOL, else 1.00.","benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,base_passthrough,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,scale=1.00 for all dates; no overlay.,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,defensive_downscale,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...","benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75,"scale=0.75 during HIGH_VOL, else 1.00.","benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,base_passthrough,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75,scale=1.00 for all dates; no overlay.,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_regime_blend_dynamic_v3,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,"benchmark_trend_regime,drawdown_regime,benchma...",Optional same-day regime overlay diagnostic; b...,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


,alpha_name,finite_pct,missing_pct,max_abs_alpha,avg_turnover_proxy,turnover_risk_flag,n_dates,n_tickers,first_valid_date,last_valid_date,status,quality_notes,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,0.980587,0.019413,3.000000,1.581392,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,0.980587,0.019413,3.000000,1.581392,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,0.980587,0.019413,3.000000,1.581392,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,0.980587,0.019413,3.000000,1.581392,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,0.980587,0.019413,3.000000,1.495814,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,0.980587,0.019413,3.000000,1.495814,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,0.980587,0.019413,3.000000,1.495814,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,0.980587,0.019413,3.000000,1.495814,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,0.980587,0.019413,3.000000,1.585623,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,0.980587,0.019413,3.000000,1.585623,LOW_TURNOVER_RISK,2088,101,2018-01-31,2026-04-23,APPROVED_FOR_REGIME_CONTEXT_WFV,"Passes overlay coverage, scale, and turnover c...",phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


,alpha_name,finite_pct,missing_pct,n_dates,n_tickers,mean_abs_alpha,alpha_std,max_abs_alpha,avg_turnover_proxy,median_turnover_proxy,max_turnover_proxy,turnover_risk_flag,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,0.980587,0.019413,2088,101,0.777126,0.992260,3.000000,1.585623,1.444444,5.783505,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.777126,0.992260,3.000000,1.585623,1.444444,5.783505,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


,alpha_name,overlay_type,base_alpha_name,scaling_rule,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,n_scaled_dates,n_total_dates,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,base_passthrough,alpha_hybrid_adaptive_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,mild_regime_scaled,alpha_hybrid_adaptive_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,defensive_downscale,alpha_hybrid_adaptive_v3,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,volatility_stress_scaled,alpha_hybrid_adaptive_v3,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,base_passthrough,alpha_rolling_ic_dynamic_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,defensive_downscale,alpha_rolling_ic_dynamic_v3,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,base_passthrough,alpha_regime_blend_dynamic_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_regime_blend_dynamic_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


## 9. Save outputs to SQLite

In [9]:
sqlite_outputs = save_regime_context_alpha_outputs(
    alpha_candidates=regime_context_alpha_candidates,
    metadata=regime_context_alpha_metadata,
    quality=regime_context_alpha_quality,
    diagnostics=regime_context_alpha_diagnostics,
    activation=regime_context_alpha_activation,
    db_path=DB_PATH,
    run_id=run_id,
    regime_context_version=REGIME_CONTEXT_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {"artifact": artifact, "current_table": current, "history_table": history, "sqlite_path": str(DB_PATH)}
        for artifact, (current, history) in REGIME_CONTEXT_ALPHA_TABLES.items()
    ]
)
display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,candidates,regime_context_alpha_candidates_current,regime_context_alpha_candidates_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,regime_context_alpha_metadata_current,regime_context_alpha_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,regime_context_alpha_quality_current,regime_context_alpha_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,diagnostics,regime_context_alpha_diagnostics_current,regime_context_alpha_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,activation,regime_context_alpha_activation_current,regime_context_alpha_activation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 10. Final Interpretation


In [10]:
quality_status_counts = (
    regime_context_alpha_quality["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_overlay_candidates")
)

notebook_05_interpretation = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "regime_context_version", "value": REGIME_CONTEXT_VERSION},
        {"metric": "base_constructed_alphas_selected", "value": len(approved_constructed_alphas)},
        {"metric": "overlay_candidates_created", "value": len(regime_context_alpha_candidates)},
        {
            "metric": "interpretation",
            "value": "Regime overlays are diagnostic candidates only and require Notebook 06 validation before use.",
        },
    ]
)

print("Notebook 05 regime overlay diagnostic interpretation")
display(notebook_05_interpretation)

print("Approved constructed alpha inputs")
display(approved_constructed_alphas[approved_display_columns])

print("Regime feature coverage")
display(regime_feature_coverage)

print("Overlay candidates created")
display(created_alpha_summary)

print("Approved/rejected quality counts")
display(quality_status_counts)

print("Diagnostics")
display(regime_context_alpha_diagnostics)

print("Activation diagnostics")
display(regime_context_alpha_activation)

print("SQLite tables written")
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving regime context alpha outputs')


Notebook 05 regime overlay diagnostic interpretation


,metric,value
0,run_id,phase5_regime_context_alpha_20260504_185103
1,regime_context_version,phase5_regime_overlay_alpha_v3
2,base_constructed_alphas_selected,9
3,overlay_candidates_created,36
4,interpretation,Regime overlays are diagnostic candidates only...


Approved constructed alpha inputs


,alpha_name,finite_pct,missing_pct,status,source_alpha_wfv_status,source_alpha_wfv_horizon,source_effective_mean_test_ic,source_persistence_ratio
0,alpha_hybrid_adaptive_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.138307,0.75
1,alpha_rolling_ic_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,20,0.133579,0.75
2,alpha_regime_blend_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045798,0.75
3,alpha_decay_aware_dynamic_v3,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,APPROVED_CONSTRUCTED_ALPHA_WFV,5,0.045317,0.75
4,alpha_smooth_regime_weighted_v2,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.147428,0.50
5,alpha_health_weighted_research_v1,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.146375,0.50
6,alpha_equal_weight_research_v1,0.980587,0.019413,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.145156,0.50
7,alpha_persistence_blend_v2,0.981066,0.018934,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.139545,0.50
8,alpha_diversified_research_v2,0.988729,0.011271,APPROVED_FOR_ALPHA_VALIDATION,WATCHLIST_CONSTRUCTED_ALPHA_WFV,20,0.125316,0.50


Regime feature coverage


...

,alpha_name,finite_pct,missing_pct,n_dates,n_tickers,mean_abs_alpha,alpha_std,max_abs_alpha,avg_turnover_proxy,median_turnover_proxy,max_turnover_proxy,turnover_risk_flag,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,0.980587,0.019413,2088,101,0.779241,0.994154,3.000000,1.581392,1.445545,5.777778,LOW_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.205939,0.948515,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,0.980587,0.019413,2088,101,0.776453,0.991431,3.000000,1.495814,1.321782,6.747475,LOW_TURNOVER_RISK,0.330460,0.917385,0.75,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,0.980587,0.019413,2088,101,0.777126,0.992260,3.000000,1.585623,1.444444,5.783505,LOW_TURNOVER_RISK,0.000000,1.000000,1.00,1.0,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,0.980587,0.019413,2088,101,0.777126,0.992260,3.000000,1.585623,1.444444,5.783505,LOW_TURNOVER_RISK,1.000000,1.025670,0.90,1.1,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


Activation diagnostics


,alpha_name,overlay_type,base_alpha_name,scaling_rule,overlay_active_pct,avg_scale_factor,min_scale_factor,max_scale_factor,n_scaled_dates,n_total_dates,run_id,regime_context_version
0,alpha_hybrid_adaptive_v3__base_passthrough,base_passthrough,alpha_hybrid_adaptive_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
1,alpha_hybrid_adaptive_v3__mild_regime_scaled,mild_regime_scaled,alpha_hybrid_adaptive_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
2,alpha_hybrid_adaptive_v3__defensive_downscale,defensive_downscale,alpha_hybrid_adaptive_v3,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
3,alpha_hybrid_adaptive_v3__volatility_stress_sc...,volatility_stress_scaled,alpha_hybrid_adaptive_v3,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
4,alpha_rolling_ic_dynamic_v3__base_passthrough,base_passthrough,alpha_rolling_ic_dynamic_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
5,alpha_rolling_ic_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_rolling_ic_dynamic_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
6,alpha_rolling_ic_dynamic_v3__defensive_downscale,defensive_downscale,alpha_rolling_ic_dynamic_v3,"scale=0.75 during HIGH_DRAWDOWN or DOWNTREND, ...",0.205939,0.948515,0.75,1.0,430,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
7,alpha_rolling_ic_dynamic_v3__volatility_stress...,volatility_stress_scaled,alpha_rolling_ic_dynamic_v3,"scale=0.75 during HIGH_VOL, else 1.00.",0.330460,0.917385,0.75,1.0,690,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
8,alpha_regime_blend_dynamic_v3__base_passthrough,base_passthrough,alpha_regime_blend_dynamic_v3,scale=1.00 for all dates; no overlay.,0.000000,1.000000,1.00,1.0,0,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3
9,alpha_regime_blend_dynamic_v3__mild_regime_scaled,mild_regime_scaled,alpha_regime_blend_dynamic_v3,scale=1.10 when not DOWNTREND/HIGH_DRAWDOWN/HI...,1.000000,1.025670,0.90,1.1,2088,2088,phase5_regime_context_alpha_20260504_185103,phase5_regime_overlay_alpha_v3


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,candidates,regime_context_alpha_candidates_current,regime_context_alpha_candidates_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,regime_context_alpha_metadata_current,regime_context_alpha_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,regime_context_alpha_quality_current,regime_context_alpha_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,diagnostics,regime_context_alpha_diagnostics_current,regime_context_alpha_diagnostics_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,activation,regime_context_alpha_activation_current,regime_context_alpha_activation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [11]:
historical_downstream_readback_note = pd.DataFrame(
    [
        {
            "note": "Historical Notebook 06 regime_context_alpha_scoring_* and regime_context_alpha_wfv_* readbacks are intentionally not loaded here.",
            "reason": "Notebook 05 is now limited to optional overlay candidate generation and writes only regime_context_alpha_* construction tables.",
        }
    ]
)
display(historical_downstream_readback_note)

,note,reason
0,Historical Notebook 06 regime_context_alpha_sc...,Notebook 05 is now limited to optional overlay...
